<a href="https://colab.research.google.com/github/akul-bharadwaj/gpt2-tokenization-memory-inference-benchmark/blob/main/Tokenization_GPT_2__Analysis_and_GPU_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenization, GPT-2 Analysis, and GPU Inference

Akul Bharadwaj Bangalore Harish

**Pre-flight Check**

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")

print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
  print(f"Device: {torch.cuda.get_device_name(0)}")
  print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch: 2.11.0+cu128
CUDA Available: True
Device: Tesla T4
VRAM: 15.64 GB


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14910 MiB


## **Section 0 : Tokenization Trade-offs**

In [ ]:
# get the corpus (book)
import os
import requests

if not os.path.exists("pg11.txt"):
    url = (
        "https://www.gutenberg.org/cache/epub/11/pg11.txt"
    )
    file_path = "pg11.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [ ]:
raw_text = open("pg11.txt", "r", encoding="utf-8").read()
print(raw_text[:100])

The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of an


### **Character-level**


In [ ]:
class CharTokenizer:

  def __init__(self, vocab):
    self.vocab = {v:k for k, v in enumerate(vocab)}
    self.inv_vocab = {v: k for k, v in self.vocab.items()}
    print(self.vocab)

  def encode(self, text):
    return [self.vocab[c] for c in list(text)]


  def decode(self, ids):
    return ''.join(self.inv_vocab[i] for i in ids)

In [ ]:
char_vocab = sorted(set(raw_text)) # To get all the unique characters

character_tokenizer = CharTokenizer(char_vocab)

char_token_ids = character_tokenizer.encode(raw_text)
decoded_text_char = character_tokenizer.decode(char_token_ids)

print("Sample Token IDs:", char_token_ids[:50])
print("Total tokens:", len(char_token_ids))
print("Sample Decoded Text: ", decoded_text_char[:50])
print("Decoded correctly:", decoded_text_char == raw_text)

{'\n': 0, ' ': 1, '!': 2, '#': 3, '$': 4, '%': 5, "'": 6, '(': 7, ')': 8, '*': 9, '+': 10, ',': 11, '-': 12, '.': 13, '/': 14, '0': 15, '1': 16, '2': 17, '3': 18, '4': 19, '5': 20, '6': 21, '7': 22, '8': 23, '9': 24, ':': 25, ';': 26, '?': 27, 'A': 28, 'B': 29, 'C': 30, 'D': 31, 'E': 32, 'F': 33, 'G': 34, 'H': 35, 'I': 36, 'J': 37, 'K': 38, 'L': 39, 'M': 40, 'N': 41, 'O': 42, 'P': 43, 'Q': 44, 'R': 45, 'S': 46, 'T': 47, 'U': 48, 'V': 49, 'W': 50, 'X': 51, 'Y': 52, 'Z': 53, '[': 54, ']': 55, '_': 56, 'a': 57, 'b': 58, 'c': 59, 'd': 60, 'e': 61, 'f': 62, 'g': 63, 'h': 64, 'i': 65, 'j': 66, 'k': 67, 'l': 68, 'm': 69, 'n': 70, 'o': 71, 'p': 72, 'q': 73, 'r': 74, 's': 75, 't': 76, 'u': 77, 'v': 78, 'w': 79, 'x': 80, 'y': 81, 'z': 82, 'ù': 83, '—': 84, '‘': 85, '’': 86, '“': 87, '”': 88, '•': 89, '™': 90}
Sample Token IDs: [47, 64, 61, 1, 43, 74, 71, 66, 61, 59, 76, 1, 34, 77, 76, 61, 70, 58, 61, 74, 63, 1, 61, 29, 71, 71, 67, 1, 71, 62, 1, 28, 68, 65, 59, 61, 6, 75, 1, 28, 60, 78, 61, 70, 7

### **Whitespace**

In [ ]:
import re

class WhitespaceTokenizer:

    def __init__(self, vocab):
        self.vocab = {token: idx for idx, token in enumerate(vocab)}
        self.inv_vocab = {idx: token for token, idx in self.vocab.items()}

    def tokenize(self, text):
        pattern = r"(\s+|--|[.,!?;:'\"()\[\]{}-]|[^\s.,!?;:'\"()\[\]{}-]+)"
        return re.findall(pattern, text)

    def encode(self, text):
        tokens = self.tokenize(text)
        return [self.vocab[token] for token in tokens]

    def decode(self, ids):
        return "".join(self.inv_vocab[i] for i in ids)

In [ ]:
temp_tokenizer = WhitespaceTokenizer([])
tokens_ws = temp_tokenizer.tokenize(raw_text)

vocab_ws = sorted(set(tokens_ws))

whitespace_tokenizer = WhitespaceTokenizer(vocab_ws)

ws_token_ids = whitespace_tokenizer.encode(raw_text)
decoded_text_ws = whitespace_tokenizer.decode(ws_token_ids)

print("Sample Token IDs:", ws_token_ids[:50])
print("Total tokens:", len(ws_token_ids))
print("Sample Decoded Text: ", decoded_text_ws[:50])
print("Decoded correctly:", decoded_text_ws == raw_text)

Sample Token IDs: [513, 22, 427, 22, 244, 22, 1422, 22, 2396, 22, 92, 33, 2842, 22, 87, 22, 1942, 22, 580, 14, 521, 22, 1422, 22, 1992, 22, 1669, 22, 3244, 22, 3445, 22, 2396, 22, 788, 22, 790, 22, 1942, 22, 3244, 22, 544, 22, 486, 22, 771, 0, 2288, 22]
Total tokens: 65702
Sample Decoded Text:  The Project Gutenberg eBook of Alice's Adventures 
Decoded correctly: True


### **Tiktoken (GPT-2 BPE)**

In [ ]:
import tiktoken
BPEtokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
BPE_token_ids = BPEtokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})
decoded_text_BPE = BPEtokenizer.decode(BPE_token_ids)

print("Sample Token IDs:", BPE_token_ids[:50])
print("Total tokens:", len(BPE_token_ids))
print("Sample Decoded Text: ", decoded_text_BPE[:50])
print("Decoded correctly:", decoded_text_BPE == raw_text)

Sample Token IDs: [464, 4935, 20336, 46566, 286, 14862, 338, 15640, 287, 42713, 198, 220, 220, 220, 220, 198, 1212, 46566, 318, 329, 262, 779, 286, 2687, 6609, 287, 262, 1578, 1829, 290, 198, 1712, 584, 3354, 286, 262, 995, 379, 645, 1575, 290, 351, 2048, 645, 8733, 198, 10919, 15485, 13, 921]
Total tokens: 49115
Sample Decoded Text:  The Project Gutenberg eBook of Alice's Adventures 
Decoded correctly: True


### **Metrics Computation for each Tokenizer**

In [ ]:
# -------------------------
# Character-level tokenizer
# -------------------------

CR_char = len(raw_text) / len(char_token_ids)

avg_token_length_char = sum(len(token) for token in char_vocab) / len(char_vocab)

vocab_size_char = len(char_vocab)

print("Compression Ratio (Character-level):", CR_char)
print("Average Token Length (Character-level):", avg_token_length_char)
print("Vocabulary Size (Character-level):", vocab_size_char)

Compression Ratio (Character-level): 1.0
Average Token Length (Character-level): 1.0
Vocabulary Size (Character-level): 91


In [ ]:
# -------------------------
# Whitespace tokenizer
# -------------------------

CR_ws = len(raw_text) / len(ws_token_ids)

avg_token_length_ws = sum(len(token) for token in vocab_ws) / len(vocab_ws)

vocab_size_ws = len(vocab_ws)

print("Compression Ratio (Whitespace):", CR_ws)
print("Average Token Length (Whitespace):", avg_token_length_ws)
print("Vocabulary Size (Whitespace):", vocab_size_ws)

Compression Ratio (Whitespace): 2.4953578277678
Average Token Length (Whitespace): 6.253650086612224
Vocabulary Size (Whitespace): 4041


In [ ]:
# -------------------------
# Tiktoken GPT-2 BPE
# -------------------------

CR_BPE = len(raw_text) / len(BPE_token_ids)

vocab_size_BPE = BPEtokenizer.n_vocab

bpe_token_lengths = []

for token_id in range(BPEtokenizer.n_vocab):
    token_bytes = BPEtokenizer.decode_single_token_bytes(token_id)
    token_str = token_bytes.decode("utf-8", errors="replace")
    bpe_token_lengths.append(len(token_str))

avg_token_length_BPE = sum(bpe_token_lengths) / len(bpe_token_lengths)

print("Compression Ratio (BPE):", CR_BPE)
print("Average Token Length (BPE):", avg_token_length_BPE)
print("Vocabulary Size (BPE):", vocab_size_BPE)

Compression Ratio (BPE): 3.3380840883640435
Average Token Length (BPE): 6.354378494538074
Vocabulary Size (BPE): 50257


### **Bar chart - Comparision on vocabulary size and compression ratio**

*Prompt*

I need to plot a bar chart using Plotly to compare the three tokenizers on vocabulary size and compression ratio.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare metrics
metrics_df = pd.DataFrame({
    "Tokenizer": ["Character-level", "Whitespace", "GPT-2 BPE"],
    "Vocabulary Size": [vocab_size_char, vocab_size_ws, vocab_size_BPE],
    "Compression Ratio": [CR_char, CR_ws, CR_BPE]
})

metrics_df

,Tokenizer,Vocabulary Size,Compression Ratio
0,Character-level,91,1.000000
1,Whitespace,4041,2.495358
2,GPT-2 BPE,50257,3.338084


In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Vocabulary Size", "Compression Ratio")
)

# Vocabulary size bar chart
fig.add_trace(
    go.Bar(
        x=metrics_df["Tokenizer"],
        y=metrics_df["Vocabulary Size"],
        name="Vocabulary Size",
        text=metrics_df["Vocabulary Size"],
        textposition="auto"
    ),
    row=1,
    col=1
)

# Compression ratio bar chart
fig.add_trace(
    go.Bar(
        x=metrics_df["Tokenizer"],
        y=metrics_df["Compression Ratio"],
        name="Compression Ratio",
        text=metrics_df["Compression Ratio"].round(2),
        textposition="auto"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Tokenizer Comparison: Vocabulary Size vs Compression Ratio",
    height=500,
    width=1000,
    showlegend=False
)

fig.update_yaxes(title_text="Vocabulary Size", row=1, col=1)
fig.update_yaxes(title_text="Characters per Token", row=1, col=2)

fig.show()

### **Analysis**

From my results, the trade-off is clear: a smaller vocabulary leads to longer token sequences, while a larger vocabulary can represent more text per token. The character-level tokenizer has only **91** vocabulary items, but its compression ratio is **1.0**, meaning every character becomes one token. The whitespace tokenizer increases vocabulary size to **4041** and improves compression to **2.50**, while GPT-2 BPE has **50257** tokens and achieves **3.34**.

From an information-theory perspective, better tokenization captures repeated patterns using fewer symbols. Although character-level tokenization preserves all information, it creates longer sequences, increasing computation and making it harder for the model to learn word-level and semantic patterns efficiently.

## **Section 1 : Model Parameters : Inference Memory Requirements**

### **Task 1: Parameter Census**

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

models = {
    "Small (124M)":  {"L": 12, "d_model": 768, "heads": 12, "d_ff": 3072, "V": 50257, "T": 1024},
    "Medium (355M)": {"L": 24, "d_model": 1024, "heads": 16, "d_ff": 4096, "V": 50257, "T": 1024},
    "Large (774M)": {"L": 36, "d_model": 1280, "heads": 20, "d_ff": 5120, "V": 50257, "T": 1024},
    "XL (1.5B)":    {"L": 48, "d_model": 1600, "heads": 25, "d_ff": 6400, "V": 50257, "T": 1024},
}

def fmt_count(x):
    return f"{x:,}"

def fmt_mb(x):
    return f"{x:,.2f} MB"

values = {}

for model_name, cfg in models.items():
    L = cfg["L"]
    d_model = cfg["d_model"]
    d_ff = cfg["d_ff"]
    V = cfg["V"]
    T = cfg["T"]

    token = V * d_model
    positional = T * d_model

    attention_qkv = 3 * (d_model ** 2)
    attention_out = d_model ** 2
    mlp_up = d_model * d_ff
    mlp_down = d_ff * d_model
    layernorms = 4 * d_model

    total_per_layer = (
        attention_qkv
        + attention_out
        + mlp_up
        + mlp_down
        + layernorms
    )

    all_layers = total_per_layer * L
    total_params = token + positional + all_layers

    fp32_memory_mb = (total_params * 4) / 1000000
    fp16_memory_mb = (total_params * 2) / 1000000

    values[model_name] = {
        "L": L,
        "Token": token,
        "Positional": positional,
        "Attention QKV": attention_qkv,
        "Attention Out": attention_out,
        "MLP Up": mlp_up,
        "MLP Down": mlp_down,
        "LayerNorms": layernorms,
        "Total Per Layer": total_per_layer,
        "All Layers": all_layers,
        "TOTAL PARAMS": total_params,
        "FP32 Memory": fp32_memory_mb,
        "FP16 Memory": fp16_memory_mb,
    }

In [ ]:
rows = [
    {
        "Component": "**Embeddings**",
        "Formula": "",
        "Small (124M)": "",
        "Medium (355M)": "",
        "Large (774M)": "",
        "XL (1.5B)": "",
    },
    {
        "Component": "Token",
        "Formula": r"$V \times d_{model}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["Token"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["Token"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["Token"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["Token"]),
    },
    {
        "Component": "Positional",
        "Formula": r"$T \times d_{model}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["Positional"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["Positional"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["Positional"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["Positional"]),
    },
    {
        "Component": "**Per Layer**",
        "Formula": "",
        "Small (124M)": r"$\times 12$",
        "Medium (355M)": r"$\times 24$",
        "Large (774M)": r"$\times 36$",
        "XL (1.5B)": r"$\times 48$",
    },
    {
        "Component": "Attention QKV",
        "Formula": r"$3 \times d_{model}^{2}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["Attention QKV"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["Attention QKV"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["Attention QKV"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["Attention QKV"]),
    },
    {
        "Component": "Attention Out",
        "Formula": r"$d_{model}^{2}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["Attention Out"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["Attention Out"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["Attention Out"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["Attention Out"]),
    },
    {
        "Component": "MLP Up",
        "Formula": r"$d_{model} \times d_{ff}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["MLP Up"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["MLP Up"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["MLP Up"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["MLP Up"]),
    },
    {
        "Component": "MLP Down",
        "Formula": r"$d_{ff} \times d_{model}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["MLP Down"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["MLP Down"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["MLP Down"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["MLP Down"]),
    },
    {
        "Component": "LayerNorms (2 LNs)",
        "Formula": r"$4 \times d_{model}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["LayerNorms"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["LayerNorms"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["LayerNorms"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["LayerNorms"]),
    },
    {
        "Component": "**Total Per Layer**",
        "Formula": "-",
        "Small (124M)": fmt_count(values["Small (124M)"]["Total Per Layer"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["Total Per Layer"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["Total Per Layer"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["Total Per Layer"]),
    },
    {
        "Component": "**All Layers**",
        "Formula": r"$L \times \text{Total Per Layer}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["All Layers"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["All Layers"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["All Layers"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["All Layers"]),
    },
    {
        "Component": "**TOTAL PARAMS**",
        "Formula": r"$\text{Token} + \text{Positional} + \text{All Layers}$",
        "Small (124M)": fmt_count(values["Small (124M)"]["TOTAL PARAMS"]),
        "Medium (355M)": fmt_count(values["Medium (355M)"]["TOTAL PARAMS"]),
        "Large (774M)": fmt_count(values["Large (774M)"]["TOTAL PARAMS"]),
        "XL (1.5B)": fmt_count(values["XL (1.5B)"]["TOTAL PARAMS"]),
    },
    {
        "Component": "**FP32 Memory**",
        "Formula": r"$\text{TOTAL PARAMS} \times 4 \div 1{,}000{,}000$",
        "Small (124M)": fmt_mb(values["Small (124M)"]["FP32 Memory"]),
        "Medium (355M)": fmt_mb(values["Medium (355M)"]["FP32 Memory"]),
        "Large (774M)": fmt_mb(values["Large (774M)"]["FP32 Memory"]),
        "XL (1.5B)": fmt_mb(values["XL (1.5B)"]["FP32 Memory"]),
    },
    {
        "Component": "**FP16 Memory**",
        "Formula": r"$\text{TOTAL PARAMS} \times 2 \div 1{,}000{,}000$",
        "Small (124M)": fmt_mb(values["Small (124M)"]["FP16 Memory"]),
        "Medium (355M)": fmt_mb(values["Medium (355M)"]["FP16 Memory"]),
        "Large (774M)": fmt_mb(values["Large (774M)"]["FP16 Memory"]),
        "XL (1.5B)": fmt_mb(values["XL (1.5B)"]["FP16 Memory"]),
    },
]

df_params = pd.DataFrame(rows)

display(Markdown(df_params.to_markdown(index=False)))

| Component           | Formula                                                | Small (124M)   | Medium (355M)   | Large (774M)   | XL (1.5B)     |
|:--------------------|:-------------------------------------------------------|:---------------|:----------------|:---------------|:--------------|
| **Embeddings**      |                                                        |                |                 |                |               |
| Token               | $V \times d_{model}$                                   | 38,597,376     | 51,463,168      | 64,328,960     | 80,411,200    |
| Positional          | $T \times d_{model}$                                   | 786,432        | 1,048,576       | 1,310,720      | 1,638,400     |
| **Per Layer**       |                                                        | $\times 12$    | $\times 24$     | $\times 36$    | $\times 48$   |
| Attention QKV       | $3 \times d_{model}^{2}$                               | 1,769,472      | 3,145,728       | 4,915,200      | 7,680,000     |
| Attention Out       | $d_{model}^{2}$                                        | 589,824        | 1,048,576       | 1,638,400      | 2,560,000     |
| MLP Up              | $d_{model} \times d_{ff}$                              | 2,359,296      | 4,194,304       | 6,553,600      | 10,240,000    |
| MLP Down            | $d_{ff} \times d_{model}$                              | 2,359,296      | 4,194,304       | 6,553,600      | 10,240,000    |
| LayerNorms (2 LNs)  | $4 \times d_{model}$                                   | 3,072          | 4,096           | 5,120          | 6,400         |
| **Total Per Layer** | -                                                      | 7,080,960      | 12,587,008      | 19,665,920     | 30,726,400    |
| **All Layers**      | $L \times \text{Total Per Layer}$                      | 84,971,520     | 302,088,192     | 707,973,120    | 1,474,867,200 |
| **TOTAL PARAMS**    | $\text{Token} + \text{Positional} + \text{All Layers}$ | 124,355,328    | 354,599,936     | 773,612,800    | 1,556,916,800 |
| **FP32 Memory**     | $\text{TOTAL PARAMS} \times 4 \div 1{,}000{,}000$      | 497.42 MB      | 1,418.40 MB     | 3,094.45 MB    | 6,227.67 MB   |
| **FP16 Memory**     | $\text{TOTAL PARAMS} \times 2 \div 1{,}000{,}000$      | 248.71 MB      | 709.20 MB       | 1,547.23 MB    | 3,113.83 MB   |

### **Task 2: Activation Memory Analysis**

In [ ]:
# Peak activation memory for the Small model (in MB)

B = 1
T = 1024
d_model = 768
H = 12
d_ff = 3072

input_embeddings = B * T * d_model
qkv = B * 3 * T * d_model
attention_scores = B * H * T * T
mlp_activations = B * T * d_ff

# 2 LayerNorm inputs per transformer block
layernorm_inputs = 2 * B * T * d_model

total_activation_elements = (
    input_embeddings
    + qkv
    + attention_scores
    + mlp_activations
    + layernorm_inputs
)

peak_activation_memory_MB = (total_activation_elements * 4) / 1000000 #FP32

print("Input embeddings:", input_embeddings)
print("QKV projections:", qkv)
print("Attention scores:", attention_scores)
print("MLP activations:", mlp_activations)
print("LayerNorm inputs:", layernorm_inputs)

print("Total activation elements:", total_activation_elements)
print("Peak activation memory:", peak_activation_memory_MB, "MB")

Input embeddings: 786432
QKV projections: 2359296
Attention scores: 12582912
MLP activations: 3145728
LayerNorm inputs: 1572864
Total activation elements: 20447232
Peak activation memory: 81.788928 MB


In [ ]:
# Ratio of activation memory to parameter memory for each model size

activation_results = {}

for model_name, cfg in models.items():
    B = 1
    T = cfg["T"]
    d_model = cfg["d_model"]
    H = cfg["heads"]
    d_ff = cfg["d_ff"]

    input_embeddings = B * T * d_model
    qkv = B * 3 * T * d_model
    attention_scores = B * H * T * T
    mlp_activations = B * T * d_ff
    layernorm_inputs = 2 * B * T * d_model

    total_activation_elements = (
        input_embeddings
        + qkv
        + attention_scores
        + mlp_activations
        + layernorm_inputs
    )

    activation_memory_MB = (total_activation_elements * 4) / 1000000

    parameter_memory_MB = values[model_name]["FP32 Memory"]

    activation_to_parameter_ratio = activation_memory_MB / parameter_memory_MB

    activation_results[model_name] = {
        "Activation Memory (MB)": activation_memory_MB,
        "Parameter Memory (MB)": parameter_memory_MB,
        "Activation / Parameter Ratio": activation_to_parameter_ratio,
        "Activation / Parameter Ratio (%)": activation_to_parameter_ratio * 100
    }

In [ ]:
df_activation_ratio = pd.DataFrame(activation_results).T

df_activation_ratio["Activation Memory (MB)"] = df_activation_ratio["Activation Memory (MB)"].map(lambda x: f"{x:.2f}")
df_activation_ratio["Parameter Memory (MB)"] = df_activation_ratio["Parameter Memory (MB)"].map(lambda x: f"{x:.2f}")
df_activation_ratio["Activation / Parameter Ratio"] = df_activation_ratio["Activation / Parameter Ratio"].map(lambda x: f"{x:.4f}")
df_activation_ratio["Activation / Parameter Ratio (%)"] = df_activation_ratio["Activation / Parameter Ratio (%)"].map(lambda x: f"{x:.2f}%")

df_activation_ratio

,Activation Memory (MB),Parameter Memory (MB),Activation / Parameter Ratio,Activation / Parameter Ratio (%)
Small (124M),81.79,497.42,0.1644,16.44%
Medium (355M),109.05,1418.40,0.0769,7.69%
Large (774M),136.31,3094.45,0.0441,4.41%
XL (1.5B),170.39,6227.67,0.0274,2.74%


**At what model size do activations become negligible compared to parameters ( < 5%)?**

For inference with batch size 1 and sequence length 1024, the attention score tensor is the dominant activation memory component. This is because attention scores have shape B × H × T × T, so their memory grows quadratically with sequence length, O(T²). In contrast, input embeddings, QKV projections, MLP activations, and LayerNorm inputs scale linearly with sequence length.

For the Small model, attention scores require 12,582,912 values, which is larger than QKV projections, MLP activations, input embeddings, and LayerNorm inputs. The same pattern holds for the larger models as well. The activation-to-parameter memory ratio decreases as model size increases: Small is around 16.44%, Medium is around 7.69%, Large is around 4.41%, and XL is around 2.74%. Therefore, activations become negligible compared to parameters from the Large model onwards, since the ratio falls below 5%.

### **Task 3: Scaling Laws**

In [ ]:
reference_params = {
    "Small (124M)": 124439808,
    "Medium (355M)": 354823168,
    "Large (774M)": 773903616
}

verification_rows = []

for model_name, reference in reference_params.items():
    calculated = values[model_name]["TOTAL PARAMS"]
    fp16_memory = values[model_name]["FP16 Memory"]

    absolute_difference = abs(calculated - reference)
    percentage_difference = (absolute_difference / reference) * 100

    verification_rows.append({
        "Model": model_name,
        "Calculated Params": calculated,
        "Reference Params": reference,
        "Absolute Difference": absolute_difference,
        "Percentage Difference (%)": percentage_difference,
        "Within ±0.1%": percentage_difference <= 0.1,
        "FP16 Memory (MB)": fp16_memory
    })

df_verify = pd.DataFrame(verification_rows)

df_verify_display = df_verify.copy()

df_verify_display["Calculated Params"] = df_verify_display["Calculated Params"].map(lambda x: f"{x:,}")
df_verify_display["Reference Params"] = df_verify_display["Reference Params"].map(lambda x: f"{x:,}")
df_verify_display["Absolute Difference"] = df_verify_display["Absolute Difference"].map(lambda x: f"{x:,}")
df_verify_display["Percentage Difference (%)"] = df_verify_display["Percentage Difference (%)"].map(lambda x: f"{x:.4f}%")
df_verify_display["FP16 Memory (MB)"] = df_verify_display["FP16 Memory (MB)"].map(lambda x: f"{x:.2f}")

df_verify_display

,Model,Calculated Params,Reference Params,Absolute Difference,Percentage Difference (%),Within ±0.1%,FP16 Memory (MB)
0,Small (124M),"124,355,328","124,439,808","84,480",0.0679%,True,248.71
1,Medium (355M),"354,599,936","354,823,168","223,232",0.0629%,True,709.20
2,Large (774M),"773,612,800","773,903,616","290,816",0.0376%,True,1547.23


In [ ]:
import numpy as np

x = df_verify["Calculated Params"]
y = df_verify["FP16 Memory (MB)"]

log_x = np.log10(x)
log_y = np.log10(y)

slope, intercept = np.polyfit(log_x, log_y, 1)

print("Log-log slope:", slope)

Log-log slope: 0.9999999999999972


In [ ]:
# Scaling plot: parameter count (x-axis) vs. FP16 memory in MB (y-axis) on a log-log scale

import plotly.express as px

slope, intercept = np.polyfit(np.log10(x), np.log10(y), 1)

fig = px.scatter(
    df_verify,
    x="Calculated Params",
    y="FP16 Memory (MB)",
    text="Model",
    log_x=True,
    log_y=True,
    title=f"Parameter Count vs FP16 Memory on Log-Log Scale (Slope ≈ {slope:.2f})",
    labels={
        "Calculated Params": "Parameter Count",
        "FP16 Memory (MB)": "FP16 Memory (MB)"
    }
)

fig.update_traces(
    mode="markers+text",
    textposition="top center"
)

fig.update_layout(
    width=800,
    height=500
)

fig.show()

**Analysis**

From my calculated FP16 memory values, Medium uses about **2.85×** the memory of Small, while Large uses about **2.18×** the memory of Medium. Although both transitions add 12 layers, the relative increase is different. Going from Small to Medium increases the number of layers from 12 to 24, which is a 2× increase. Going from Medium to Large increases layers from 24 to 36, which is only a 1.5× increase.

The scaling is also affected by the embedding matrices. Token and positional embeddings are not repeated for every transformer layer; they exist once for the model. Therefore, only the transformer block parameters scale directly with the number of layers. Since the fixed embedding portion becomes a smaller fraction as the model grows, total memory does not increase in a simple “+12 layers means same multiplier” way.

This shows sub-linear scaling between Medium and Large: adding 12 layers has a smaller proportional impact on an already deeper model than it does on the smaller model.

## **Section 2: From CPU to GPU : Inference Profiling & Optimization**

### **Task 1: CPU Baseline**

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import time
import gc
import psutil
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
device = torch.device("cpu")

model_names = {
    "Small 124M": "gpt2",
    "Medium 355M": "gpt2-medium",
    "Large 774M": "gpt2-large"
}

BATCH_SIZE = 1
INPUT_SEQ_LEN = 512
NEW_TOKENS = 128

process = psutil.Process(os.getpid())

In [ ]:
def get_ram_mb():
    return process.memory_info().rss / 1_000_000


def benchmark_model(model_label, hf_model_name, raw_text):
    print(f"\nRunning {model_label}: {hf_model_name}")

    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    model = AutoModelForCausalLM.from_pretrained(hf_model_name)
    model.to(device)
    model.eval()

    # Used text from the book and truncate to 512 tokens
    encoded = tokenizer(
        raw_text,
        return_tensors="pt",
        truncation=True,
        max_length=INPUT_SEQ_LEN
    )

    input_ids = encoded["input_ids"].to(device)

    ram_after_load = get_ram_mb()

    # -------------------------
    # Warm-up forward passes
    # -------------------------
    with torch.no_grad():
        for _ in range(2):
            _ = model(input_ids, use_cache=True)

    gc.collect()

    # -------------------------
    # Timed autoregressive generation
    # -------------------------
    token_latencies = []
    peak_ram_mb = get_ram_mb()

    generated_ids = input_ids.clone()

    with torch.no_grad():
        start_total = time.perf_counter()

        # First token: full 512-token context
        start = time.perf_counter()

        outputs = model(input_ids, use_cache=True)
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
        past_key_values = outputs.past_key_values

        end = time.perf_counter()

        first_token_latency = end - start
        token_latencies.append(first_token_latency)

        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        peak_ram_mb = max(peak_ram_mb, get_ram_mb())

        # Remaining 127 tokens: use KV cache
        current_token = next_token

        for _ in range(NEW_TOKENS - 1):
            start = time.perf_counter()

            outputs = model(
                current_token,
                past_key_values=past_key_values,
                use_cache=True
            )

            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            past_key_values = outputs.past_key_values

            end = time.perf_counter()

            token_latencies.append(end - start)

            generated_ids = torch.cat([generated_ids, next_token], dim=1)
            current_token = next_token

            peak_ram_mb = max(peak_ram_mb, get_ram_mb())

        end_total = time.perf_counter()

    total_generation_time = end_total - start_total

    # Tokens 10 to 50 inclusive, using 1-based token numbering
    steady_state_latencies = token_latencies[9:50]
    steady_state_latency = sum(steady_state_latencies) / len(steady_state_latencies)

    throughput = NEW_TOKENS / total_generation_time

    result = {
        "Model": model_label,
        "HF Model": hf_model_name,
        "Input Sequence Length": INPUT_SEQ_LEN,
        "Generated Tokens": NEW_TOKENS,
        "Cold-start Latency / TTFT (s)": first_token_latency,
        "Steady-state Latency / ITL tokens 10-50 (s)": steady_state_latency,
        "Throughput (tokens/sec)": throughput,
        "RAM After Model Load (MB)": ram_after_load,
        "Peak RAM During Generation (MB)": peak_ram_mb
    }

    # Clean up before next model
    del model
    del tokenizer
    del input_ids
    del generated_ids
    del outputs
    gc.collect()

    return result

In [ ]:
results = []

for model_label, hf_model_name in model_names.items():
    result = benchmark_model(model_label, hf_model_name, raw_text)
    results.append(result)

df_cpu_inference = pd.DataFrame(results)
df_cpu_inference


Running Small 124M: gpt2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

,Model,HF Model,Input Sequence Length,Generated Tokens,Cold-start Latency / TTFT (s),Steady-state Latency / ITL tokens 10-50 (s),Throughput (tokens/sec),RAM After Model Load (MB),Peak RAM During Generation (MB)
0,Small 124M,gpt2,512,128,2.370184,0.121106,9.695032,1124.278272,2078.040064
1,Medium 355M,gpt2-medium,512,128,4.149166,0.167799,4.876560,1233.264640,3324.137472
2,Large 774M,gpt2-large,512,128,8.453334,0.348968,2.398740,1158.782976,5491.744768


In [ ]:
df_cpu_display = pd.DataFrame({
    "Model": ["Small 124M", "Medium 355M", "Large 774M"],

    "First Token Latency": df_cpu_inference["Cold-start Latency / TTFT (s)"],

    "Throughput (tok/s)": df_cpu_inference["Throughput (tokens/sec)"],

    "RAM Used": df_cpu_inference["Peak RAM During Generation (MB)"]
})

df_cpu_display["First Token Latency"] = df_cpu_display["First Token Latency"].map(lambda x: f"{x:.2f} s")
df_cpu_display["Throughput (tok/s)"] = df_cpu_display["Throughput (tok/s)"].map(lambda x: f"{x:.2f}")
df_cpu_display["RAM Used"] = df_cpu_display["RAM Used"].map(lambda x: f"{x:.2f} MB")

df_cpu_display

,Model,First Token Latency,Throughput (tok/s),RAM Used
0,Small 124M,2.37 s,9.70,2078.04 MB
1,Medium 355M,4.15 s,4.88,3324.14 MB
2,Large 774M,8.45 s,2.40,5491.74 MB


### **Task 2: GPU Migration & Basic Acceleration**

In [ ]:
device = torch.device("cuda")

In [ ]:
def benchmark_model_gpu(model_label, hf_model_name, raw_text):
    print(f"\nRunning {model_label} on GPU: {hf_model_name}")

    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    model = AutoModelForCausalLM.from_pretrained(hf_model_name)

    # -------------------------
    # Device transfer timing
    # CPU RAM -> GPU VRAM
    # -------------------------
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_transfer = time.time()
    model = model.cuda()
    torch.cuda.synchronize()
    end_transfer = time.time()

    device_transfer_time = end_transfer - start_transfer

    model.eval()

    encoded = tokenizer(
        raw_text,
        return_tensors="pt",
        truncation=True,
        max_length=INPUT_SEQ_LEN
    )

    input_ids = encoded["input_ids"].cuda()

    # -------------------------
    # Warm-up forward passes
    # -------------------------
    with torch.no_grad():
        for _ in range(2):
            _ = model(input_ids, use_cache=True)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    # -------------------------
    # Timed autoregressive generation
    # -------------------------
    generated_ids = input_ids.clone()

    with torch.no_grad():
        torch.cuda.synchronize()
        start_total = time.time()

        # First token: full 512-token context
        outputs = model(input_ids, use_cache=True)
        next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
        past_key_values = outputs.past_key_values

        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        current_token = next_token

        # Remaining 127 tokens using KV cache
        for _ in range(NEW_TOKENS - 1):
            outputs = model(
                current_token,
                past_key_values=past_key_values,
                use_cache=True
            )

            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            past_key_values = outputs.past_key_values

            generated_ids = torch.cat([generated_ids, next_token], dim=1)
            current_token = next_token

        torch.cuda.synchronize()
        end_total = time.time()

    gpu_generation_time = end_total - start_total
    gpu_throughput = NEW_TOKENS / gpu_generation_time

    peak_vram_mb = torch.cuda.max_memory_allocated() / 1_000_000

    result = {
        "Model": model_label,
        "Device Transfer Time (s)": device_transfer_time,
        "GPU Generation Time (s)": gpu_generation_time,
        "GPU Throughput (tok/s)": gpu_throughput,
        "Peak VRAM Usage (MB)": peak_vram_mb
    }

    del model
    del tokenizer
    del input_ids
    del generated_ids
    del outputs
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
gpu_results = []

for model_label, hf_model_name in model_names.items():
    result = benchmark_model_gpu(model_label, hf_model_name, raw_text)
    gpu_results.append(result)

df_gpu = pd.DataFrame(gpu_results)
df_gpu


Running Small 124M on GPU: gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Running Medium 355M on GPU: gpt2-medium


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]


Running Large 774M on GPU: gpt2-large


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

,Model,Device Transfer Time (s),GPU Generation Time (s),GPU Throughput (tok/s),Peak VRAM Usage (MB)
0,Small 124M,2.158941,1.273921,100.477206,800.854016
1,Medium 355M,5.234152,2.060121,62.132278,1839.194112
2,Large 774M,7.707065,3.098541,41.309761,3767.789568


In [ ]:
df_gpu["CPU Generation Time (s)"] = NEW_TOKENS / df_cpu_inference["Throughput (tokens/sec)"]

df_gpu["Speedup Ratio (CPU/GPU)"] = (
    df_gpu["CPU Generation Time (s)"] / df_gpu["GPU Generation Time (s)"]
)

In [ ]:
df_gpu_display = df_gpu.copy()

df_gpu_display["Device Transfer Time (s)"] = df_gpu_display["Device Transfer Time (s)"].map(lambda x: f"{x:.2f} s")
df_gpu_display["GPU Generation Time (s)"] = df_gpu_display["GPU Generation Time (s)"].map(lambda x: f"{x:.2f} s")
df_gpu_display["GPU Throughput (tok/s)"] = df_gpu_display["GPU Throughput (tok/s)"].map(lambda x: f"{x:.2f}")
df_gpu_display["Peak VRAM Usage (MB)"] = df_gpu_display["Peak VRAM Usage (MB)"].map(lambda x: f"{x:.2f} MB")
df_gpu_display["CPU Generation Time (s)"] = df_gpu_display["CPU Generation Time (s)"].map(lambda x: f"{x:.2f} s")
df_gpu_display["Speedup Ratio (CPU/GPU)"] = df_gpu_display["Speedup Ratio (CPU/GPU)"].map(lambda x: f"{x:.2f}x")

df_gpu_display

,Model,Device Transfer Time (s),GPU Generation Time (s),GPU Throughput (tok/s),Peak VRAM Usage (MB),CPU Generation Time (s),Speedup Ratio (CPU/GPU)
0,Small 124M,2.16 s,1.27 s,100.48,800.85 MB,13.20 s,10.36x
1,Medium 355M,5.23 s,2.06 s,62.13,1839.19 MB,26.25 s,12.74x
2,Large 774M,7.71 s,3.10 s,41.31,3767.79 MB,53.36 s,17.22x


**Does speedup scale linearly with model size? Why does the Large model hit higher throughput while Small only manages lower throughput gains?**

No, the speedup does **not scale linearly** with model size. From my measured results, speedup increases from **10.36× for Small**, to **12.74× for Medium**, and **17.22× for Large**, but the increase is not proportional to the parameter count or model size.

The main reason is **fixed GPU overhead**. For smaller models, overheads such as kernel launch time, synchronization, device scheduling, and autoregressive token-by-token decoding form a larger share of total runtime. Because the Small model has less computation per forward pass, the GPU is not fully utilized, so the relative gain is limited.

For the Large model, each forward pass involves much larger matrix multiplications, which are better suited to GPU parallelism. The CPU generation time increases sharply from **13.20s** for Small to **53.36s** for Large, while GPU generation time increases more moderately from **1.27s** to **3.10s**. This produces a higher relative speedup for the Large model.

One important nuance is that Large does **not** have higher absolute GPU throughput. Small reaches **100.48 tok/s**, while Large reaches **41.31 tok/s**. Therefore, Large has higher **relative speedup**, not higher raw throughput.

### **Task 3: Optimization Experiments**

In [ ]:
# FP32 GPU baseline for Medium 355M using 3 warm-up runs and 5 averaged timed runs

device = torch.device("cuda")

MODEL_NAME = "gpt2-medium"
MODEL_LABEL = "Medium 355M"
INPUT_SEQ_LEN = 512
NEW_TOKENS = 128

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


def safe_reset_peak_memory():
    try:
        torch.cuda.reset_peak_memory_stats(device)
    except RuntimeError:
        print("Could not reset peak memory stats. Continuing...")


def prepare_input(tokenizer, raw_text):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    encoded = tokenizer(
        raw_text,
        return_tensors="pt",
        truncation=True,
        max_length=INPUT_SEQ_LEN,
        padding="max_length"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    return input_ids, attention_mask


def compute_perplexity(model, tokenizer, text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss.float()
        perplexity = torch.exp(loss).item()

    return perplexity


def benchmark_generation(model, tokenizer, input_ids, attention_mask):
    model.eval()

    torch.cuda.empty_cache()
    torch.cuda.synchronize(device)

    # 3 warm-up generation runs
    with torch.no_grad():
        for _ in range(3):
            _ = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

    torch.cuda.synchronize(device)

    generation_times = []
    throughputs = []
    peak_vram_values = []

    # 5 timed generation runs
    for _ in range(5):
        torch.cuda.empty_cache()
        safe_reset_peak_memory()
        torch.cuda.synchronize(device)

        with torch.no_grad():
            start_time = time.time()

            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

            torch.cuda.synchronize(device)
            end_time = time.time()

        generation_time = end_time - start_time
        generated_tokens = output_ids.shape[1] - input_ids.shape[1]
        throughput = generated_tokens / generation_time
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1_000_000

        generation_times.append(generation_time)
        throughputs.append(throughput)
        peak_vram_values.append(peak_vram_mb)

        del output_ids

    avg_generation_time = sum(generation_times) / len(generation_times)
    avg_throughput = sum(throughputs) / len(throughputs)
    max_peak_vram = max(peak_vram_values)

    return avg_generation_time, avg_throughput, max_peak_vram


# Load tokenizer and FP32 model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
input_ids, attention_mask = prepare_input(tokenizer, raw_text)

torch.cuda.empty_cache()
gc.collect()

model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_fp32.eval()

# Benchmark FP32 generation
fp32_latency, fp32_throughput, fp32_vram = benchmark_generation(
    model_fp32,
    tokenizer,
    input_ids,
    attention_mask
)

# FP32 perplexity baseline
test_sentence = "Alice was beginning to get very tired of sitting by her sister on the bank."
fp32_perplexity = compute_perplexity(model_fp32, tokenizer, test_sentence)

# Store result
df_medium_fp32_baseline = pd.DataFrame([{
    "Model": MODEL_LABEL,
    "Device": "T4 GPU",
    "Precision": "FP32",
    "Batch Size": 1,
    "Generation Latency (s)": fp32_latency,
    "Throughput (tok/s)": fp32_throughput,
    "Peak VRAM (MB)": fp32_vram,
    "Perplexity": fp32_perplexity
}])

df_medium_fp32_baseline_display = df_medium_fp32_baseline.copy()

df_medium_fp32_baseline_display["Generation Latency (s)"] = df_medium_fp32_baseline_display["Generation Latency (s)"].map(lambda x: f"{x:.2f}")
df_medium_fp32_baseline_display["Throughput (tok/s)"] = df_medium_fp32_baseline_display["Throughput (tok/s)"].map(lambda x: f"{x:.2f}")
df_medium_fp32_baseline_display["Peak VRAM (MB)"] = df_medium_fp32_baseline_display["Peak VRAM (MB)"].map(lambda x: f"{x:.2f}")
df_medium_fp32_baseline_display["Perplexity"] = df_medium_fp32_baseline_display["Perplexity"].map(lambda x: f"{x:.4f}")

df_medium_fp32_baseline_display

CUDA available: True
GPU: Tesla T4


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


,Model,Device,Precision,Batch Size,Generation Latency (s),Throughput (tok/s),Peak VRAM (MB),Perplexity
0,Medium 355M,T4 GPU,FP32,1,2.32,55.71,1577.65,29.0545


#### **Option A: Mixed Precision (FP16)**

In [ ]:
# Get FP32 baseline values from the previous baseline dataframe
fp32_latency = df_medium_fp32_baseline.loc[0, "Generation Latency (s)"]
fp32_throughput = df_medium_fp32_baseline.loc[0, "Throughput (tok/s)"]
fp32_vram = df_medium_fp32_baseline.loc[0, "Peak VRAM (MB)"]
fp32_perplexity = df_medium_fp32_baseline.loc[0, "Perplexity"]

# Clear FP32 model from GPU if it is still loaded
try:
    del model_fp32
except:
    pass

gc.collect()
torch.cuda.empty_cache()

# Load Medium model in FP16
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_fp16 = model_fp16.half().to(device)
model_fp16.eval()

# Benchmark FP16 generation using the same 3 warm-up + 5 timed-run method
fp16_latency, fp16_throughput, fp16_vram = benchmark_generation(
    model_fp16,
    tokenizer,
    input_ids,
    attention_mask
)

# Compute FP16 perplexity on the same test sentence
fp16_perplexity = compute_perplexity(
    model_fp16,
    tokenizer,
    test_sentence
)

# Calculate comparison metrics
fp16_speedup = fp16_throughput / fp32_throughput
vram_reduction = ((fp32_vram - fp16_vram) / fp32_vram) * 100
perplexity_change = ((fp16_perplexity - fp32_perplexity) / fp32_perplexity) * 100

# Create before/after table
df_option_a = pd.DataFrame([
    {
        "Model": MODEL_LABEL,
        "Precision": "FP32 Baseline",
        "Batch Size": 1,
        "Generation Latency (s)": fp32_latency,
        "Throughput (tok/s)": fp32_throughput,
        "Peak VRAM (MB)": fp32_vram,
        "Perplexity": fp32_perplexity,
        "Speedup vs FP32": 1.0,
        "VRAM Reduction vs FP32 (%)": 0.0,
        "Perplexity Change vs FP32 (%)": 0.0
    },
    {
        "Model": MODEL_LABEL,
        "Precision": "FP16 Mixed Precision",
        "Batch Size": 1,
        "Generation Latency (s)": fp16_latency,
        "Throughput (tok/s)": fp16_throughput,
        "Peak VRAM (MB)": fp16_vram,
        "Perplexity": fp16_perplexity,
        "Speedup vs FP32": fp16_speedup,
        "VRAM Reduction vs FP32 (%)": vram_reduction,
        "Perplexity Change vs FP32 (%)": perplexity_change
    }
])

# Clean display version
df_option_a_display = df_option_a.copy()

df_option_a_display["Generation Latency (s)"] = df_option_a_display["Generation Latency (s)"].map(lambda x: f"{x:.2f}")
df_option_a_display["Throughput (tok/s)"] = df_option_a_display["Throughput (tok/s)"].map(lambda x: f"{x:.2f}")
df_option_a_display["Peak VRAM (MB)"] = df_option_a_display["Peak VRAM (MB)"].map(lambda x: f"{x:.2f}")
df_option_a_display["Perplexity"] = df_option_a_display["Perplexity"].map(lambda x: f"{x:.4f}")
df_option_a_display["Speedup vs FP32"] = df_option_a_display["Speedup vs FP32"].map(lambda x: f"{x:.2f}x")
df_option_a_display["VRAM Reduction vs FP32 (%)"] = df_option_a_display["VRAM Reduction vs FP32 (%)"].map(lambda x: f"{x:.2f}%")
df_option_a_display["Perplexity Change vs FP32 (%)"] = df_option_a_display["Perplexity Change vs FP32 (%)"].map(lambda x: f"{x:.2f}%")

df_option_a_display

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

,Model,Precision,Batch Size,Generation Latency (s),Throughput (tok/s),Peak VRAM (MB),Perplexity,Speedup vs FP32,VRAM Reduction vs FP32 (%),Perplexity Change vs FP32 (%)
0,Medium 355M,FP32 Baseline,1,2.32,55.71,1577.65,29.0545,1.00x,0.00%,0.00%
1,Medium 355M,FP16 Mixed Precision,1,2.82,49.10,806.24,29.1146,0.88x,48.90%,0.21%


In [ ]:
print("FP16 speedup vs FP32:", f"{fp16_speedup:.2f}x")
print("VRAM reduction:", f"{vram_reduction:.2f}%")
print("FP32 perplexity:", f"{fp32_perplexity:.4f}")
print("FP16 perplexity:", f"{fp16_perplexity:.4f}")
print("Perplexity change:", f"{perplexity_change:.2f}%")
print("Within 1% perplexity difference:", abs(perplexity_change) <= 1)

FP16 speedup vs FP32: 0.88x
VRAM reduction: 48.90%
FP32 perplexity: 29.0545
FP16 perplexity: 29.1146
Perplexity change: 0.21%
Within 1% perplexity difference: True


#### **Option B: Batch Size Scaling**

In [ ]:
import plotly.express as px

batch_sizes = [1, 2, 4, 8, 16]

# Clear FP16 model from GPU if it is still loaded
try:
    del model_fp16
except:
    pass

gc.collect()
torch.cuda.empty_cache()

# Load Medium model again in FP32 for batch-size scaling
model_fp32_batch = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_fp32_batch.eval()

# Prepare a single input sequence, then repeat it for different batch sizes
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

encoded = tokenizer(
    raw_text,
    return_tensors="pt",
    truncation=True,
    max_length=INPUT_SEQ_LEN,
    padding="max_length"
)

single_input_ids = encoded["input_ids"].to(device)
single_attention_mask = encoded["attention_mask"].to(device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [ ]:
def benchmark_batch_size(model, tokenizer, single_input_ids, single_attention_mask, batch_size):
    input_ids = single_input_ids.repeat(batch_size, 1)
    attention_mask = single_attention_mask.repeat(batch_size, 1)

    torch.cuda.empty_cache()
    torch.cuda.synchronize(device)

    # 3 warm-up generation runs
    with torch.no_grad():
        for _ in range(3):
            _ = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

    torch.cuda.synchronize(device)

    generation_times = []
    throughputs = []
    latency_per_token_steps = []
    latency_per_output_tokens = []
    peak_vram_values = []

    # 5 timed generation runs
    for _ in range(5):
        torch.cuda.empty_cache()
        safe_reset_peak_memory()
        torch.cuda.synchronize(device)

        with torch.no_grad():
            start_time = time.time()

            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

            torch.cuda.synchronize(device)
            end_time = time.time()

        generation_time = end_time - start_time
        total_generated_tokens = batch_size * NEW_TOKENS

        throughput = total_generated_tokens / generation_time
        latency_per_token_step = generation_time / NEW_TOKENS
        latency_per_output_token = generation_time / total_generated_tokens
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1_000_000

        generation_times.append(generation_time)
        throughputs.append(throughput)
        latency_per_token_steps.append(latency_per_token_step)
        latency_per_output_tokens.append(latency_per_output_token)
        peak_vram_values.append(peak_vram_mb)

        del output_ids

    avg_generation_time = sum(generation_times) / len(generation_times)
    avg_throughput = sum(throughputs) / len(throughputs)
    avg_latency_per_token_step = sum(latency_per_token_steps) / len(latency_per_token_steps)
    avg_latency_per_output_token = sum(latency_per_output_tokens) / len(latency_per_output_tokens)
    max_peak_vram = max(peak_vram_values)

    del input_ids
    del attention_mask

    return {
        "Batch Size": batch_size,
        "Generation Latency (s)": avg_generation_time,
        "Throughput (tok/s)": avg_throughput,
        "Latency per Token Step (s)": avg_latency_per_token_step,
        "Latency per Output Token (s)": avg_latency_per_output_token,
        "Peak VRAM (MB)": max_peak_vram
    }

In [ ]:
option_b_results = []

for batch_size in batch_sizes:
    try:
        print(f"Running Medium 355M FP32 with batch size {batch_size}...")

        result = benchmark_batch_size(
            model_fp32_batch,
            tokenizer,
            single_input_ids,
            single_attention_mask,
            batch_size
        )

        result["Model"] = MODEL_LABEL
        result["Precision"] = "FP32"
        result["Baseline Throughput (tok/s)"] = fp32_throughput
        result["Baseline VRAM (MB)"] = fp32_vram
        result["Throughput Speedup vs Baseline"] = result["Throughput (tok/s)"] / fp32_throughput
        result["VRAM Change vs Baseline (%)"] = ((result["Peak VRAM (MB)"] - fp32_vram) / fp32_vram) * 100

        option_b_results.append(result)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM at batch size {batch_size}. Stopping here.")
            print(e)
            torch.cuda.empty_cache()
            break
        else:
            raise e

df_option_b = pd.DataFrame(option_b_results)
df_option_b

Running Medium 355M FP32 with batch size 1...
Running Medium 355M FP32 with batch size 2...
Running Medium 355M FP32 with batch size 4...
Running Medium 355M FP32 with batch size 8...
Running Medium 355M FP32 with batch size 16...


,Batch Size,Generation Latency (s),Throughput (tok/s),Latency per Token Step (s),Latency per Output Token (s),Peak VRAM (MB),Model,Precision,Baseline Throughput (tok/s),Baseline VRAM (MB),Throughput Speedup vs Baseline,VRAM Change vs Baseline (%)
0,1,3.243211,43.622118,0.025338,0.025338,1577.662464,Medium 355M,FP32,55.714687,1577.64608,0.782955,0.001039
1,2,3.663572,80.152392,0.028622,0.014311,1723.561984,Medium 355M,FP32,55.714687,1577.64608,1.438622,9.248963
2,4,4.129260,136.911878,0.032260,0.008065,2017.196032,Medium 355M,FP32,55.714687,1577.64608,2.457375,27.861125
3,8,5.689992,182.512777,0.044453,0.005557,2604.464128,Medium 355M,FP32,55.714687,1577.64608,3.275847,65.085450
4,16,9.560658,215.738846,0.074693,0.004668,3779.000320,Medium 355M,FP32,55.714687,1577.64608,3.872208,139.534099


In [ ]:
df_option_b_display = df_option_b.copy()

df_option_b_display["Generation Latency (s)"] = df_option_b_display["Generation Latency (s)"].map(lambda x: f"{x:.2f}")
df_option_b_display["Throughput (tok/s)"] = df_option_b_display["Throughput (tok/s)"].map(lambda x: f"{x:.2f}")
df_option_b_display["Latency per Token Step (s)"] = df_option_b_display["Latency per Token Step (s)"].map(lambda x: f"{x:.4f}")
df_option_b_display["Latency per Output Token (s)"] = df_option_b_display["Latency per Output Token (s)"].map(lambda x: f"{x:.4f}")
df_option_b_display["Peak VRAM (MB)"] = df_option_b_display["Peak VRAM (MB)"].map(lambda x: f"{x:.2f}")
df_option_b_display["Throughput Speedup vs Baseline"] = df_option_b_display["Throughput Speedup vs Baseline"].map(lambda x: f"{x:.2f}x")
df_option_b_display["VRAM Change vs Baseline (%)"] = df_option_b_display["VRAM Change vs Baseline (%)"].map(lambda x: f"{x:.2f}%")

df_option_b_display

,Batch Size,Generation Latency (s),Throughput (tok/s),Latency per Token Step (s),Latency per Output Token (s),Peak VRAM (MB),Model,Precision,Baseline Throughput (tok/s),Baseline VRAM (MB),Throughput Speedup vs Baseline,VRAM Change vs Baseline (%)
0,1,3.24,43.62,0.0253,0.0253,1577.66,Medium 355M,FP32,55.714687,1577.64608,0.78x,0.00%
1,2,3.66,80.15,0.0286,0.0143,1723.56,Medium 355M,FP32,55.714687,1577.64608,1.44x,9.25%
2,4,4.13,136.91,0.0323,0.0081,2017.20,Medium 355M,FP32,55.714687,1577.64608,2.46x,27.86%
3,8,5.69,182.51,0.0445,0.0056,2604.46,Medium 355M,FP32,55.714687,1577.64608,3.28x,65.09%
4,16,9.56,215.74,0.0747,0.0047,3779.00,Medium 355M,FP32,55.714687,1577.64608,3.87x,139.53%


In [ ]:
# Saturation Point

df_option_b = df_option_b.sort_values("Batch Size").reset_index(drop=True)
df_option_b["Throughput Gain vs Previous (%)"] = df_option_b["Throughput (tok/s)"].pct_change() * 100

gain_threshold = 10
saturation_batch = None
saturation_label = None

for i in range(1, len(df_option_b)):
    gain = df_option_b.loc[i, "Throughput Gain vs Previous (%)"]

    if gain < gain_threshold:
        saturation_batch = df_option_b.loc[i - 1, "Batch Size"]
        saturation_label = f"Saturation ≈ BS {saturation_batch}"
        break

if saturation_batch is None:
    saturation_batch = df_option_b.loc[df_option_b["Throughput (tok/s)"].idxmax(), "Batch Size"]
    saturation_label = f"Best tested BS {saturation_batch}"

print(saturation_label)

Best tested BS 16


In [ ]:
fig = px.line(
    df_option_b,
    x="Batch Size",
    y="Throughput (tok/s)",
    markers=True,
    title="Medium 355M FP32 Batch Size Scaling: Throughput vs Batch Size",
    labels={
        "Batch Size": "Batch Size",
        "Throughput (tok/s)": "Throughput (tokens/sec)"
    }
)

fig.add_vline(
    x=saturation_batch,
    line_dash="dash",
    annotation_text=saturation_label,
    annotation_position="top left"
)

fig.update_layout(
    width=850,
    height=500
)

fig.show()

**Observation**

In [ ]:
# Approximate FLOPs utilization for the best tested batch size

t4_peak_tflops = 65  # T4 FP16 Tensor Core theoretical peak

medium_params = values["Medium (355M)"]["TOTAL PARAMS"]

best_row = df_option_b.loc[df_option_b["Throughput (tok/s)"].idxmax()]
best_batch_size = best_row["Batch Size"]
best_throughput = best_row["Throughput (tok/s)"]

# Rough decoder-only inference estimate
flops_per_token = 2 * medium_params

achieved_tflops = (flops_per_token * best_throughput) / 1e12
percent_of_t4_peak = (achieved_tflops / t4_peak_tflops) * 100

print(f"Best batch size: {best_batch_size}")
print(f"Best throughput: {best_throughput:.2f} tok/s")
print(f"Estimated achieved TFLOPs: {achieved_tflops:.4f}")
print(f"Approx. percent of T4 theoretical peak: {percent_of_t4_peak:.4f}%")

Best batch size: 16
Best throughput: 215.74 tok/s
Estimated achieved TFLOPs: 0.1530
Approx. percent of T4 theoretical peak: 0.2354%


At the best tested batch size, throughput was highest, but this should be interpreted as the best tested configuration rather than a guaranteed saturation point if throughput continued increasing up to batch size 16. Larger batches improve throughput by increasing GPU parallelism, but they also increase per-request latency and VRAM usage.

**One failure mode**

In [ ]:
import traceback

large_model = None
large_tokenizer = None
single_input_ids = None
single_attention_mask = None
input_ids = None
attention_mask = None
output_ids = None

oom_error_message = None
oom_batch_size = None

try:
    large_tokenizer = AutoTokenizer.from_pretrained("gpt2-large")
    large_tokenizer.pad_token = large_tokenizer.eos_token
    large_tokenizer.padding_side = "left"

    large_model = AutoModelForCausalLM.from_pretrained("gpt2-large").to(device)
    large_model.eval()

    encoded = large_tokenizer(
        raw_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    single_input_ids = encoded["input_ids"].to(device)
    single_attention_mask = encoded["attention_mask"].to(device)

    # Required batch size 32 first.
    # If batch size 32 succeeds, continue increasing until OOM is observed.
    for batch_size in [32, 64, 128, 256]:
        try:
            print(f"\nTrying GPT-2 Large with batch size {batch_size}...")

            input_ids = single_input_ids.repeat(batch_size, 1)
            attention_mask = single_attention_mask.repeat(batch_size, 1)

            torch.cuda.empty_cache()

            try:
                torch.cuda.reset_peak_memory_stats(device)
            except RuntimeError:
                pass

            with torch.no_grad():
                torch.cuda.synchronize(device)

                output_ids = large_model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=128,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=large_tokenizer.eos_token_id
                )

                torch.cuda.synchronize(device)

            peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1_000_000

            print(f"Batch size {batch_size} completed successfully.")
            print(f"Peak VRAM usage: {peak_vram_mb:.2f} MB")

            del input_ids
            del attention_mask
            del output_ids
            input_ids = None
            attention_mask = None
            output_ids = None

            gc.collect()
            torch.cuda.empty_cache()

        except RuntimeError:
            oom_batch_size = batch_size
            oom_error_message = traceback.format_exc()

            print(f"\nOOM / RuntimeError encountered at batch size {batch_size}:")
            print(oom_error_message)

            break

finally:
    del large_model
    del large_tokenizer
    del single_input_ids
    del single_attention_mask
    del input_ids
    del attention_mask
    del output_ids

    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]


Trying GPT-2 Large with batch size 32...
Batch size 32 completed successfully.
Peak VRAM usage: 12487.99 MB

Trying GPT-2 Large with batch size 64...

OOM / RuntimeError encountered at batch size 64:
Traceback (most recent call last):
  File "/tmp/ipykernel_2903/3723120757.py", line 52, in <cell line: 0>
    output_ids = large_model.generate(
                 ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2582, in generate
    result = decoding_method(
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2782, in _sample
    outputs = self._prefill(
              ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 3828, in _prefill
    ret

For GPT-2 Large, batch size 32 completed successfully in my Colab T4 environment with peak VRAM usage of 12.49 GB. Since no OOM occurred at batch size 32, I increased the batch size to 64 to demonstrate the failure mode. At batch size 64, CUDA ran out of memory, and the full OOM traceback is shown above.

### **Final Comparison Matrix**

In [ ]:
final_fp16_configs = [
    {
        "Configuration": "Small 124M",
        "HF Model": "gpt2",
        "Batch": 1
    },
    {
        "Configuration": "Medium 355M",
        "HF Model": "gpt2-medium",
        "Batch": 8
    },
    {
        "Configuration": "Large 774M",
        "HF Model": "gpt2-large",
        "Batch": 4
    }
]


def benchmark_final_fp16_config(hf_model_name, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    encoded = tokenizer(
        raw_text,
        return_tensors="pt",
        truncation=True,
        max_length=INPUT_SEQ_LEN,
        padding="max_length"
    )

    single_input_ids = encoded["input_ids"].to(device)
    single_attention_mask = encoded["attention_mask"].to(device)

    input_ids = single_input_ids.repeat(batch_size, 1)
    attention_mask = single_attention_mask.repeat(batch_size, 1)

    torch.cuda.empty_cache()
    gc.collect()

    model = AutoModelForCausalLM.from_pretrained(hf_model_name)
    model = model.half().to(device)
    model.eval()

    torch.cuda.empty_cache()
    torch.cuda.synchronize(device)

    # 3 warm-up generation runs
    with torch.no_grad():
        for _ in range(3):
            _ = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

    torch.cuda.synchronize(device)

    generation_times = []
    throughputs = []
    peak_vram_values = []

    # 5 timed generation runs
    for _ in range(5):
        torch.cuda.empty_cache()
        safe_reset_peak_memory()
        torch.cuda.synchronize(device)

        with torch.no_grad():
            start_time = time.time()

            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

            torch.cuda.synchronize(device)
            end_time = time.time()

        generation_time = end_time - start_time
        total_generated_tokens = batch_size * NEW_TOKENS
        throughput = total_generated_tokens / generation_time
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1_000_000

        generation_times.append(generation_time)
        throughputs.append(throughput)
        peak_vram_values.append(peak_vram_mb)

        del output_ids

    avg_generation_time = sum(generation_times) / len(generation_times)
    avg_throughput = sum(throughputs) / len(throughputs)
    max_peak_vram_mb = max(peak_vram_values)

    del model
    del tokenizer
    del single_input_ids
    del single_attention_mask
    del input_ids
    del attention_mask

    gc.collect()
    torch.cuda.empty_cache()

    return avg_generation_time, avg_throughput, max_peak_vram_mb

In [ ]:
final_fp16_results = []

for cfg in final_fp16_configs:
    print(f"Running {cfg['Configuration']} | FP16 | Batch {cfg['Batch']}")

    latency, throughput, peak_vram_mb = benchmark_final_fp16_config(
        cfg["HF Model"],
        cfg["Batch"]
    )

    final_fp16_results.append({
        "Configuration": cfg["Configuration"],
        "Device": "T4",
        "Precision": "FP16",
        "Batch": cfg["Batch"],
        "Generation Latency (s)": latency,
        "Tok/sec": throughput,
        "VRAM (GB)": peak_vram_mb / 1000
    })

df_final_fp16_configs = pd.DataFrame(final_fp16_results)
df_final_fp16_configs

Running Small 124M | FP16 | Batch 1


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Running Medium 355M | FP16 | Batch 8


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Running Large 774M | FP16 | Batch 4


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

,Configuration,Device,Precision,Batch,Generation Latency (s),Tok/sec,VRAM (GB)
0,Small 124M,T4,FP16,1,1.158074,110.539875,1.722809
1,Medium 355M,T4,FP16,8,2.643485,388.779394,2.727834
2,Large 774M,T4,FP16,4,3.715381,138.627086,3.472334


In [ ]:
# Final Comparison Matrix: combine CPU baseline, FP32 GPU baseline, and required FP16 GPU configs

# -------------------------
# Helper functions
# -------------------------

def get_cpu_throughput(model_keyword):
    row = df_cpu_inference[
        df_cpu_inference["Model"].str.contains(model_keyword, case=False, na=False)
    ].iloc[0]
    return row["Throughput (tokens/sec)"]


def get_gpu_fp32_values(model_keyword):
    row = df_gpu[
        df_gpu["Model"].str.contains(model_keyword, case=False, na=False)
    ].iloc[0]
    throughput = row["GPU Throughput (tok/s)"]
    vram_gb = row["Peak VRAM Usage (MB)"] / 1000
    return throughput, vram_gb


def get_fp16_values(configuration, batch_size):
    row = df_final_fp16_configs[
        (df_final_fp16_configs["Configuration"] == configuration) &
        (df_final_fp16_configs["Batch"] == batch_size)
    ].iloc[0]
    throughput = row["Tok/sec"]
    vram_gb = row["VRAM (GB)"]
    return throughput, vram_gb

In [ ]:
# -------------------------
# CPU throughput baselines
# -------------------------

small_cpu_throughput = get_cpu_throughput("Small")
medium_cpu_throughput = get_cpu_throughput("Medium")
large_cpu_throughput = get_cpu_throughput("Large")

# -------------------------
# Row values
# -------------------------

small_gpu_fp32_throughput, small_gpu_fp32_vram = get_gpu_fp32_values("Small")

small_fp16_throughput, small_fp16_vram = get_fp16_values("Small 124M", 1)
medium_fp16_throughput, medium_fp16_vram = get_fp16_values("Medium 355M", 8)
large_fp16_throughput, large_fp16_vram = get_fp16_values("Large 774M", 4)

# -------------------------
# Final matrix
# -------------------------

final_matrix_rows = [
    {
        "Configuration": "Small 124M",
        "Device": "CPU",
        "Precision": "FP32",
        "Batch": 1,
        "Tok/sec": small_cpu_throughput,
        "VRAM (GB)": "N/A",
        "Speedup vs. CPU": 1.0
    },
    {
        "Configuration": "Small 124M",
        "Device": "T4",
        "Precision": "FP32",
        "Batch": 1,
        "Tok/sec": small_gpu_fp32_throughput,
        "VRAM (GB)": small_gpu_fp32_vram,
        "Speedup vs. CPU": small_gpu_fp32_throughput / small_cpu_throughput
    },
    {
        "Configuration": "Small 124M",
        "Device": "T4",
        "Precision": "FP16",
        "Batch": 1,
        "Tok/sec": small_fp16_throughput,
        "VRAM (GB)": small_fp16_vram,
        "Speedup vs. CPU": small_fp16_throughput / small_cpu_throughput
    },
    {
        "Configuration": "Medium 355M",
        "Device": "T4",
        "Precision": "FP16",
        "Batch": 8,
        "Tok/sec": medium_fp16_throughput,
        "VRAM (GB)": medium_fp16_vram,
        "Speedup vs. CPU": medium_fp16_throughput / medium_cpu_throughput
    },
    {
        "Configuration": "Large 774M",
        "Device": "T4",
        "Precision": "FP16",
        "Batch": 4,
        "Tok/sec": large_fp16_throughput,
        "VRAM (GB)": large_fp16_vram,
        "Speedup vs. CPU": large_fp16_throughput / large_cpu_throughput
    }
]

df_final_matrix = pd.DataFrame(final_matrix_rows)

df_final_matrix

,Configuration,Device,Precision,Batch,Tok/sec,VRAM (GB),Speedup vs. CPU
0,Small 124M,CPU,FP32,1,9.695032,N/A,1.000000
1,Small 124M,T4,FP32,1,100.477206,0.800854,10.363782
2,Small 124M,T4,FP16,1,110.539875,1.722809,11.401703
3,Medium 355M,T4,FP16,8,388.779394,2.727834,79.724107
4,Large 774M,T4,FP16,4,138.627086,3.472334,57.791629


In [ ]:
df_final_matrix_display = df_final_matrix.copy()

df_final_matrix_display["Tok/sec"] = df_final_matrix_display["Tok/sec"].map(lambda x: f"{x:.2f}")

df_final_matrix_display["VRAM (GB)"] = df_final_matrix_display["VRAM (GB)"].map(
    lambda x: x if x == "N/A" else f"{x:.2f}"
)

df_final_matrix_display["Speedup vs. CPU"] = df_final_matrix_display["Speedup vs. CPU"].map(lambda x: f"{x:.2f}x")

df_final_matrix_display

,Configuration,Device,Precision,Batch,Tok/sec,VRAM (GB),Speedup vs. CPU
0,Small 124M,CPU,FP32,1,9.70,N/A,1.00x
1,Small 124M,T4,FP32,1,100.48,0.80,10.36x
2,Small 124M,T4,FP16,1,110.54,1.72,11.40x
3,Medium 355M,T4,FP16,8,388.78,2.73,79.72x
4,Large 774M,T4,FP16,4,138.63,3.47,57.79x


The speedup values are computed relative to the corresponding CPU throughput for each model size, while VRAM reports peak GPU memory allocated during generation.